In [1]:
import argparse
import pandas as pd
from PIL import Image
import torch
from src.utils.naive_prompter import NaivePrompter ,read_json
api_key='INSERT OPEN AI API KEY HERE'
dataset_path='../data/dataset/filtered_dataset.csv'
images_path='../data/images/sd/reference'
df=pd.read_csv(dataset_path)

In [2]:
args = argparse.Namespace()
args.__dict__.update(read_json("../src/configs/hpme_config.json"))
args.prompt_len = 8
args.clip_model='ViT-L-14'
args.clip_pretrain='laion2b_s32b_b82k'
device = "cuda:0" if torch.cuda.is_available() else "cpu"
optimizer=NaivePrompter(args,api_key=api_key,early_stopping=True,early_stopping_patience=100)

In [ ]:
iteration=0
for idx, row in df.iterrows():
    print(f'Iteration: {iteration}')
    image_file = f"{images_path}/{idx}.png"
    image = Image.open(image_file)
    
    optimizer.args.prompt_len=8
    learned_prompt, _ = optimizer.optimize_prompt(target_images=[image])
    df.at[idx, 'soft_prompt_8_L_14'] = learned_prompt
    print(f'Soft 8 done!')

    optimizer.args.prompt_len=32
    learned_prompt, _ = optimizer.optimize_prompt(target_images=[image])
    df.at[idx, 'soft_prompt_32_L_14'] = learned_prompt
    print(f'Soft 32 done!')
    
    optimizer.args.prompt_len=64
    learned_prompt, _ = optimizer.optimize_prompt(target_images=[image])
    df.at[idx, 'soft_prompt_64_L_14'] = learned_prompt
    print(f'Soft 64 done!')
    df.to_csv(dataset_path, index=False)
    iteration+=1